In [1]:
!unzip '/content/drive/MyDrive/red/zip.zip'

Archive:  /content/drive/MyDrive/red/zip.zip
   creating: ml_alignment_solution/
   creating: ml_alignment_solution/data/
  inflating: ml_alignment_solution/data/public_test_quality.jsonl  
  inflating: ml_alignment_solution/data/kid_adult.jsonl  
  inflating: ml_alignment_solution/data/public_test_style.jsonl  
  inflating: ml_alignment_solution/data/good_bad.jsonl  
   creating: ml_alignment_solution/metrics/
  inflating: ml_alignment_solution/metrics/style_clf.pkl  
  inflating: ml_alignment_solution/УСЛОВИЯ.md  
  inflating: ml_alignment_solution/alignment_solution_colab.ipynb  
  inflating: ml_alignment_solution/requirements.txt  
  inflating: ml_alignment_solution/.gitignore  
  inflating: ml_alignment_solution/README.md  


In [2]:
%pip install -q \
  "transformers==4.53.3" \
  "peft==0.16.0" \
  "accelerate==1.9.0" \
  "bitsandbytes==0.49.2" \
  "scikit-learn==1.7.2" \
  "safetensors>=0.5.3" \
  "sentencepiece>=0.2.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 129.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.1/367.1 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 99.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
import os
import gc
import json
import math
import pickle
import random
import shutil
import subprocess
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from scipy.sparse import hstack
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
    set_seed,
)
from peft import (
    LoraConfig,
    PeftModel,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)

assert torch.cuda.is_available(), "В Colab выбери Runtime → Change runtime type → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

GPU: Tesla T4
PyTorch: 2.11.0+cu128


In [6]:
# Один раз замени строку на URL своего публичного GitHub-репозитория до финального коммита.
REPO_URL = ""  # пример: https://github.com/username/ml-alignment.git
REPO_BRANCH = "main"
CLONE_DIR = Path("/content/ml_alignment_solution")


def locate_task_root(base: Path):
    candidates = [
        base,
        base / "ml_alignment_solution",
        base / "ml-olympiad-red-task",
    ]

    if base.exists():
        candidates.extend(p for p in base.iterdir() if p.is_dir())

    for p in candidates:
        if (
            (p / "data" / "kid_adult.jsonl").exists()
            and (p / "metrics" / "style_clf.pkl").exists()
        ):
            return p.resolve()

    return None

ROOT = locate_task_root(Path.cwd())
if ROOT is None:
    ROOT = locate_task_root(Path("/content"))

if ROOT is None and REPO_URL.strip():
    shutil.rmtree(CLONE_DIR, ignore_errors=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(CLONE_DIR)],
        check=True,
    )
    ROOT = locate_task_root(CLONE_DIR)

if ROOT is None:
    raise FileNotFoundError(
        "Не найдены data/kid_adult.jsonl и metrics/style_clf.pkl. "
        "Добавь файлы в репозиторий и укажи REPO_URL."
    )

DATA_DIR = ROOT / "data"
STYLE_CLF_PATH = ROOT / "metrics" / "style_clf.pkl"
WORK_DIR = Path("/content/alignment_run")
ADAPTER_DIR = WORK_DIR / "adapters"
RESULT_DIR = WORK_DIR / "results"

shutil.rmtree(WORK_DIR, ignore_errors=True)
ADAPTER_DIR.mkdir(parents=True)
RESULT_DIR.mkdir(parents=True)

print("ROOT:", ROOT)

ROOT: /content/ml_alignment_solution


In [7]:
SEED = 42
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
MAX_LENGTH = 512
MAX_NEW_TOKENS = 192
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

RESULTS = {"seed": SEED, "base_model": BASE_MODEL}

In [8]:
def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

kid_adult_train = read_jsonl(DATA_DIR / "kid_adult.jsonl")
good_bad_train = read_jsonl(DATA_DIR / "good_bad.jsonl")
style_test = read_jsonl(DATA_DIR / "public_test_style.jsonl")
quality_test = read_jsonl(DATA_DIR / "public_test_quality.jsonl")

assert len(kid_adult_train) == 1489
assert len(good_bad_train) == 2226
assert len(style_test) == 50
assert len(quality_test) == 50

print("kid_adult train:", len(kid_adult_train))
print("good_bad train:", len(good_bad_train))
print("public style test:", len(style_test))
print("public quality test:", len(quality_test))

kid_adult train: 1489
good_bad train: 2226
public style test: 50
public quality test: 50


In [9]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


def common_prefix_length(a, b):
    n = min(len(a), len(b))
    i = 0
    while i < n and a[i] == b[i]:
        i += 1
    return i


def encode_answer(prompt, answer, max_length=MAX_LENGTH):
    user = [{"role": "user", "content": prompt}]
    dialog = user + [{"role": "assistant", "content": answer}]

    prompt_ids = tokenizer.apply_chat_template(
        user,
        tokenize=True,
        add_generation_prompt=True,
    )
    full_ids = tokenizer.apply_chat_template(
        dialog,
        tokenize=True,
        add_generation_prompt=False,
    )

    start = len(prompt_ids)
    if full_ids[:start] != prompt_ids:
        start = common_prefix_length(prompt_ids, full_ids)

    full_ids = full_ids[:max_length]
    start = min(start, len(full_ids))
    labels = [-100] * start + full_ids[start:]

    if not any(x != -100 for x in labels):
        raise ValueError("После обрезки не осталось токенов ответа")

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }


class ListDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]


class PreferenceCollator:
    def __init__(self, tok):
        self.base = DataCollatorForSeq2Seq(
            tokenizer=tok,
            padding=True,
            label_pad_token_id=-100,
            return_tensors="pt",
        )

    def __call__(self, features):
        chosen = [
            {
                "input_ids": f["chosen_input_ids"],
                "attention_mask": f["chosen_attention_mask"],
                "labels": f["chosen_labels"],
            }
            for f in features
        ]
        rejected = [
            {
                "input_ids": f["rejected_input_ids"],
                "attention_mask": f["rejected_attention_mask"],
                "labels": f["rejected_labels"],
            }
            for f in features
        ]
        batch = self.base(chosen + rejected)
        out = {
            "input_ids": batch["input_ids"],
            "attention_mask": batch["attention_mask"],
            "labels": batch["labels"],
            "pair_count": len(features),
        }
        if "ref_chosen_logp" in features[0]:
            out["ref_chosen_logp"] = torch.tensor(
                [f["ref_chosen_logp"] for f in features], dtype=torch.float32
            )
            out["ref_rejected_logp"] = torch.tensor(
                [f["ref_rejected_logp"] for f in features], dtype=torch.float32
            )
        return out


preference_collator = PreferenceCollator(tokenizer)
sft_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)


def make_pair_rows(rows, prompt_key, chosen_key, rejected_key):
    result = []
    for row in rows:
        c = encode_answer(row[prompt_key], row[chosen_key])
        r = encode_answer(row[prompt_key], row[rejected_key])
        result.append({
            "chosen_input_ids": c["input_ids"],
            "chosen_attention_mask": c["attention_mask"],
            "chosen_labels": c["labels"],
            "rejected_input_ids": r["input_ids"],
            "rejected_attention_mask": r["attention_mask"],
            "rejected_labels": r["labels"],
        })
    return result

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [10]:
def quantization_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )


def causal_lora_config():
    return LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=TARGET_MODULES,
    )


def reward_lora_config():
    return LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.SEQ_CLS,
        target_modules=TARGET_MODULES,
        modules_to_save=["score"],
    )


def load_base_causal(for_training):
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=quantization_config(),
        device_map={"": 0},
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    if for_training:
        model = prepare_model_for_kbit_training(
            model,
            use_gradient_checkpointing=True,
        )
    return model


def new_causal_model():
    model = get_peft_model(load_base_causal(for_training=True), causal_lora_config())
    model.enable_input_require_grads()
    model.print_trainable_parameters()
    return model


def load_causal_adapter(path, trainable):
    base = load_base_causal(for_training=trainable)
    model = PeftModel.from_pretrained(base, str(path), is_trainable=trainable)
    if trainable:
        model.enable_input_require_grads()
        model.print_trainable_parameters()
    return model


def response_logps(model, input_ids, attention_mask, labels):
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
    )
    logits = outputs.logits[:, :-1].float()
    shifted_labels = labels[:, 1:]
    mask = shifted_labels.ne(-100)
    safe_labels = shifted_labels.masked_fill(~mask, 0)
    token_logps = torch.gather(
        F.log_softmax(logits, dim=-1),
        dim=-1,
        index=safe_labels.unsqueeze(-1),
    ).squeeze(-1)
    sums = (token_logps * mask).sum(dim=-1)
    lengths = mask.sum(dim=-1).clamp_min(1)
    return sums, lengths


def split_pair_stats(model, batch):
    n = int(batch.pop("pair_count"))
    sums, lengths = response_logps(
        model,
        batch["input_ids"],
        batch["attention_mask"],
        batch["labels"],
    )
    return sums[:n], sums[n:], lengths[:n], lengths[n:]


def to_device(batch, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}


def compute_pair_logps(model, rows, batch_size=4, normalized=False):
    loader = DataLoader(
        ListDataset(rows),
        batch_size=batch_size,
        shuffle=False,
        collate_fn=preference_collator,
        num_workers=0,
    )
    device = next(model.parameters()).device
    chosen_all, rejected_all = [], []
    model.eval()
    with torch.inference_mode():
        for batch in loader:
            batch = to_device(batch, device)
            batch.pop("ref_chosen_logp", None)
            batch.pop("ref_rejected_logp", None)
            c, r, lc, lr = split_pair_stats(model, batch)
            if normalized:
                c = c / lc
                r = r / lr
            chosen_all.extend(c.detach().cpu().tolist())
            rejected_all.extend(r.detach().cpu().tolist())
    return chosen_all, rejected_all


def add_reference_logps(model, rows, batch_size=4):
    c, r = compute_pair_logps(model, rows, batch_size=batch_size, normalized=False)
    enriched = []
    for row, a, b in zip(rows, c, r):
        x = dict(row)
        x["ref_chosen_logp"] = float(a)
        x["ref_rejected_logp"] = float(b)
        enriched.append(x)
    return enriched


def implicit_preference_accuracy(model, rows, batch_size=4):
    c, r = compute_pair_logps(model, rows, batch_size=batch_size, normalized=True)
    return float(np.mean(np.asarray(c) > np.asarray(r)))


def cleanup_cuda():
    gc.collect()
    torch.cuda.empty_cache()

In [11]:
def train_args(name, epochs, lr, batch_size, grad_accum, warmup=0.05):
    return TrainingArguments(
        output_dir=str(WORK_DIR / f"trainer_{name}"),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        learning_rate=lr,
        lr_scheduler_type="cosine",
        warmup_ratio=warmup,
        weight_decay=0.01,
        max_grad_norm=0.3,
        logging_steps=10,
        logging_first_step=True,
        save_strategy="no",
        report_to="none",
        fp16=True,
        bf16=False,
        tf32=False,
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        remove_unused_columns=False,
        dataloader_num_workers=0,
        seed=SEED,
        data_seed=SEED,
    )


class DPOPairTrainer(Trainer):
    def __init__(self, *args, beta=0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.beta = beta

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        ref_c = inputs.pop("ref_chosen_logp")
        ref_r = inputs.pop("ref_rejected_logp")
        c, r, _, _ = split_pair_stats(model, inputs)
        logits = self.beta * ((c - r) - (ref_c - ref_r))
        loss = -F.logsigmoid(logits).mean()
        if return_outputs:
            return loss, {"chosen_logp": c.detach(), "rejected_logp": r.detach()}
        return loss


class SimPOPairTrainer(Trainer):
    def __init__(self, *args, beta=2.0, gamma_beta_ratio=0.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.beta = beta
        self.gamma_beta_ratio = gamma_beta_ratio

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        inputs.pop("ref_chosen_logp", None)
        inputs.pop("ref_rejected_logp", None)
        c, r, lc, lr = split_pair_stats(model, inputs)
        mean_c = c / lc
        mean_r = r / lr
        logits = self.beta * (mean_c - mean_r - self.gamma_beta_ratio)
        loss = -F.logsigmoid(logits).mean()
        if return_outputs:
            return loss, {"chosen_reward": mean_c.detach(), "rejected_reward": mean_r.detach()}
        return loss


class RewardPairTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        inputs.pop("ref_chosen_logp", None)
        inputs.pop("ref_rejected_logp", None)
        n = int(inputs.pop("pair_count"))
        labels = inputs.pop("labels")
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            use_cache=False,
        )
        scores = outputs.logits.squeeze(-1).float()
        chosen_scores = scores[:n]
        rejected_scores = scores[n:]
        loss = -F.logsigmoid(chosen_scores - rejected_scores).mean()
        if return_outputs:
            return loss, {
                "chosen_score": chosen_scores.detach(),
                "rejected_score": rejected_scores.detach(),
            }
        return loss

In [12]:
with open(STYLE_CLF_PATH, "rb") as f:
    style_metric = pickle.load(f)


def generate_answers(model, prompts, batch_size=4):
    old_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    answers = []
    model.eval()
    model.config.use_cache = True
    device = next(model.parameters()).device

    for start in range(0, len(prompts), batch_size):
        part = prompts[start:start + batch_size]
        texts = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": p}],
                tokenize=False,
                add_generation_prompt=True,
            )
            for p in part
        ]
        batch = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        )
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.inference_mode():
            generated = model.generate(
                **batch,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                num_beams=1,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )
        new_tokens = generated[:, batch["input_ids"].shape[1]:]
        answers.extend(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))

    model.config.use_cache = False
    tokenizer.padding_side = old_side
    return [x.strip() for x in answers]


def p_simple_values(texts):
    word_vec, char_vec = style_metric["vecs"]
    x = hstack([word_vec.transform(texts), char_vec.transform(texts)], format="csr")
    clf = style_metric["clf"]
    positive_index = int(np.where(clf.classes_ == 1)[0][0])
    return clf.predict_proba(x)[:, positive_index]


def style_interval(x):
    if x < 0.4:
        return "А"
    if x < 0.7:
        return "Б"
    if x < 0.9:
        return "В"
    return "Г"


def quality_interval(x):
    if x < 0.6:
        return "А"
    if x < 0.75:
        return "Б"
    if x < 0.9:
        return "В"
    return "Г"


def save_generations(name, prompts, answers):
    path = RESULT_DIR / f"{name}.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        for p, a in zip(prompts, answers):
            f.write(json.dumps({"prompt": p, "answer": a}, ensure_ascii=False) + "\n")
    return path

# 1

In [13]:
sft_rows = [encode_answer(row["prompt"], row["kid"]) for row in kid_adult_train]
sft_dataset = ListDataset(sft_rows)
SFT_DIR = ADAPTER_DIR / "task1_sft"

sft_model = new_causal_model()
sft_trainer = Trainer(
    model=sft_model,
    args=train_args(
        name="task1_sft",
        epochs=2.0,
        lr=2e-4,
        batch_size=2,
        grad_accum=8,
    ),
    train_dataset=sft_dataset,
    data_collator=sft_collator,
)
sft_trainer.train()
sft_model.save_pretrained(SFT_DIR)
tokenizer.save_pretrained(SFT_DIR)

style_prompts = [row["prompt"] for row in style_test]
task1_answers = generate_answers(sft_model, style_prompts, batch_size=4)
task1_probs = p_simple_values(task1_answers)
task1_score = float(task1_probs.mean())
task1_letter = style_interval(task1_score)

RESULTS["task1"] = {"metric": "P_simple", "value": task1_score, "interval": task1_letter}
save_generations("task1_sft_generations", style_prompts, task1_answers)
print(f"Задача 1: P_simple = {task1_score:.6f}; ответ = {task1_letter}")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

...

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Step,Training Loss
1,2.807000
10,2.179300
20,1.464100
30,1.366800
40,1.289600
50,1.287600
60,1.238900
70,1.240900
80,1.214000
90,1.243600


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'

Задача 1: P_simple = 0.973392; ответ = Г


In [14]:
del sft_trainer, sft_model
cleanup_cuda()

# 2

In [15]:
style_pair_rows = make_pair_rows(
    kid_adult_train,
    prompt_key="prompt",
    chosen_key="kid",
    rejected_key="adult",
)
DPO_STYLE_DIR = ADAPTER_DIR / "task2_dpo_style"

style_dpo_model = load_causal_adapter(SFT_DIR, trainable=True)
style_pair_rows_with_ref = add_reference_logps(style_dpo_model, style_pair_rows, batch_size=4)

style_dpo_trainer = DPOPairTrainer(
    model=style_dpo_model,
    args=train_args(
        name="task2_dpo_style",
        epochs=1.0,
        lr=5e-6,
        batch_size=2,
        grad_accum=8,
    ),
    train_dataset=ListDataset(style_pair_rows_with_ref),
    data_collator=preference_collator,
    beta=0.1,
)
style_dpo_trainer.train()
style_dpo_model.save_pretrained(DPO_STYLE_DIR)
tokenizer.save_pretrained(DPO_STYLE_DIR)

task2_answers = generate_answers(style_dpo_model, style_prompts, batch_size=4)
task2_probs = p_simple_values(task2_answers)
task2_score = float(task2_probs.mean())
task2_letter = style_interval(task2_score)

RESULTS["task2"] = {"metric": "P_simple", "value": task2_score, "interval": task2_letter}
save_generations("task2_dpo_style_generations", style_prompts, task2_answers)
print(f"Задача 2: P_simple = {task2_score:.6f}; ответ = {task2_letter}")
print(f"Изменение относительно SFT: {task2_score - task1_score:+.6f}")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
1,5.561100
10,4.917100


Step,Training Loss
1,5.561100
10,4.917100
20,2.002900
30,0.614100
40,0.227000
50,0.078400
60,0.047900
70,0.028200
80,0.020100
90,0.032700


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p'

Задача 2: P_simple = 0.993190; ответ = Г
Изменение относительно SFT: +0.019799


In [16]:
del style_dpo_trainer, style_dpo_model
cleanup_cuda()

# 3

In [17]:
quality_train_pairs = make_pair_rows(
    good_bad_train,
    prompt_key="instruction",
    chosen_key="chosen",
    rejected_key="rejected",
)
quality_test_pairs = make_pair_rows(
    quality_test,
    prompt_key="prompt",
    chosen_key="chosen",
    rejected_key="rejected",
)
RM_DIR = ADAPTER_DIR / "task3_reward_model"

rm_base = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=1,
    problem_type="regression",
    quantization_config=quantization_config(),
    device_map={"": 0},
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
rm_base.config.pad_token_id = tokenizer.pad_token_id
rm_base.config.use_cache = False
rm_base = prepare_model_for_kbit_training(rm_base, use_gradient_checkpointing=True)
reward_model = get_peft_model(rm_base, reward_lora_config())
reward_model.enable_input_require_grads()
reward_model.print_trainable_parameters()

rm_trainer = RewardPairTrainer(
    model=reward_model,
    args=train_args(
        name="task3_reward_model",
        epochs=2.0,
        lr=1e-4,
        batch_size=2,
        grad_accum=8,
    ),
    train_dataset=ListDataset(quality_train_pairs),
    data_collator=preference_collator,
)
rm_trainer.train()
reward_model.save_pretrained(RM_DIR)
tokenizer.save_pretrained(RM_DIR)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-4B-Instruct-2507 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


trainable params: 33,032,704 || all params: 4,055,503,360 || trainable%: 0.8145


Step,Training Loss
1,1.089200
10,0.845400
20,0.168400
30,0.014200
40,0.119100
50,0.020600
60,0.023700
70,0.012100
80,0.001200
90,0.009900


Step,Training Loss
1,1.089200
10,0.845400
20,0.168400
30,0.014200
40,0.119100
50,0.020600
60,0.023700
70,0.012100
80,0.001200
90,0.009900


('/content/alignment_run/adapters/task3_reward_model/tokenizer_config.json',
 '/content/alignment_run/adapters/task3_reward_model/special_tokens_map.json',
 '/content/alignment_run/adapters/task3_reward_model/chat_template.jinja',
 '/content/alignment_run/adapters/task3_reward_model/vocab.json',
 '/content/alignment_run/adapters/task3_reward_model/merges.txt',
 '/content/alignment_run/adapters/task3_reward_model/added_tokens.json',
 '/content/alignment_run/adapters/task3_reward_model/tokenizer.json')

In [18]:
def reward_pair_accuracy(model, rows, batch_size=4):
    loader = DataLoader(
        ListDataset(rows),
        batch_size=batch_size,
        shuffle=False,
        collate_fn=preference_collator,
        num_workers=0,
    )
    device = next(model.parameters()).device
    good = 0
    total = 0
    model.eval()
    with torch.inference_mode():
        for batch in loader:
            batch = to_device(batch, device)
            n = int(batch.pop("pair_count"))
            batch.pop("labels")
            batch.pop("ref_chosen_logp", None)
            batch.pop("ref_rejected_logp", None)
            scores = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                use_cache=False,
            ).logits.squeeze(-1).float()
            good += int((scores[:n] > scores[n:]).sum().item())
            total += n
    return good / total


task3_score = float(reward_pair_accuracy(reward_model, quality_test_pairs, batch_size=4))
task3_letter = quality_interval(task3_score)
RESULTS["task3"] = {"metric": "pairwise_accuracy", "value": task3_score, "interval": task3_letter}
print(f"Задача 3: pairwise accuracy = {task3_score:.6f}; ответ = {task3_letter}")

Задача 3: pairwise accuracy = 1.000000; ответ = Г


In [19]:
del rm_trainer, reward_model, rm_base
cleanup_cuda()

# 4

In [20]:
DPO_QUALITY_DIR = ADAPTER_DIR / "task4_dpo_quality"
quality_dpo_model = load_causal_adapter(DPO_STYLE_DIR, trainable=True)
quality_rows_with_ref = add_reference_logps(quality_dpo_model, quality_train_pairs, batch_size=4)

quality_dpo_trainer = DPOPairTrainer(
    model=quality_dpo_model,
    args=train_args(
        name="task4_dpo_quality",
        epochs=1.0,
        lr=5e-6,
        batch_size=2,
        grad_accum=8,
    ),
    train_dataset=ListDataset(quality_rows_with_ref),
    data_collator=preference_collator,
    beta=0.1,
)
quality_dpo_trainer.train()
quality_dpo_model.save_pretrained(DPO_QUALITY_DIR)
tokenizer.save_pretrained(DPO_QUALITY_DIR)

task4_score = implicit_preference_accuracy(quality_dpo_model, quality_test_pairs, batch_size=4)
task4_letter = quality_interval(task4_score)
RESULTS["task4"] = {
    "metric": "implicit_preference_accuracy_mean_logprob",
    "value": task4_score,
    "interval": task4_letter,
}
print(f"Задача 4: implicit-preference accuracy = {task4_score:.6f}; ответ = {task4_letter}")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Step,Training Loss
1,5.482800
10,5.046500
20,3.291100
30,2.207600
40,1.778900
50,1.436100
60,1.374600
70,1.048400
80,0.982300
90,1.150800


Задача 4: implicit-preference accuracy = 0.980000; ответ = Г


In [21]:
del quality_dpo_trainer, quality_dpo_model
cleanup_cuda()

# 5

In [22]:
SIMPO_DIR = ADAPTER_DIR / "task5_simpo"
simpo_model = load_causal_adapter(DPO_STYLE_DIR, trainable=True)

simpo_trainer = SimPOPairTrainer(
    model=simpo_model,
    args=train_args(
        name="task5_simpo",
        epochs=2.0,
        lr=1e-6,
        batch_size=2,
        grad_accum=8,
    ),
    train_dataset=ListDataset(quality_train_pairs),
    data_collator=preference_collator,
    beta=2.0,
    gamma_beta_ratio=0.5,
)
simpo_trainer.train()
simpo_model.save_pretrained(SIMPO_DIR)
tokenizer.save_pretrained(SIMPO_DIR)

task5_score = implicit_preference_accuracy(simpo_model, quality_test_pairs, batch_size=4)
task5_letter = quality_interval(task5_score)
RESULTS["task5"] = {
    "metric": "implicit_preference_accuracy_mean_logprob",
    "value": task5_score,
    "interval": task5_letter,
}
print(f"Задача 5: implicit-preference accuracy = {task5_score:.6f}; ответ = {task5_letter}")
print(f"Разница SimPO − DPO quality: {task5_score - task4_score:+.6f}")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


Step,Training Loss
1,5.064900
10,5.304300
20,5.815800
30,5.055700
40,5.378800
50,5.446600
60,5.533800
70,5.059600
80,5.077200
90,5.696200


Step,Training Loss
1,5.064900
10,5.304300
20,5.815800
30,5.055700
40,5.378800
50,5.446600
60,5.533800
70,5.059600
80,5.077200
90,5.696200


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.63 GiB. GPU 0 has a total capacity of 14.56 GiB of which 603.81 MiB is free. Including non-PyTorch memory, this process has 13.93 GiB memory in use. Of the allocated memory 12.50 GiB is allocated by PyTorch, and 1.24 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
del simpo_trainer, simpo_model
cleanup_cuda()